In [ ]:
!pip install -q datasets transformers evaluate scikit-learn accelerate


In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, pipeline
)
from datasets import load_dataset, DatasetDict
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from huggingface_hub import login
print("Đã import xong")

In [ ]:
from huggingface_hub import login
login()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from datasets import load_dataset

dataset = load_dataset("visolex/ViSFD")
print(dataset)

In [ ]:
print(dataset['train'][0])


In [ ]:
print(dataset['train'].column_names)

In [ ]:
import json
sample = dataset['train'][0]
print(json.dumps(sample, ensure_ascii=False, indent=2))

In [ ]:
# Đếm số lượng mẫu theo n_star (số sao)
from collections import Counter

stars = [item['n_star'] for item in dataset['train']]
counter = Counter(stars)
print("Phân bố theo số sao:")
for star, count in sorted(counter.items()):
    print(f"  {star} sao: {count} mẫu")

In [ ]:
for i in range(5):
    item = dataset['train'][i]
    print(f"--- Mẫu {i} ---")
    print(f"Comment: {item['comment']}")
    print(f"Số sao: {item['n_star']}")
    print()

In [ ]:
def map_star_to_sentiment(n_star):
    if n_star <= 2:
        return 'NEGATIVE'
    elif n_star == 3:
        return 'NEUTRAL'
    else:
        return 'POSITIVE'

# Kiểm tra
for star in [1, 2, 3, 4, 5]:
    print(f"{star} sao → {map_star_to_sentiment(star)}")

In [ ]:
def add_sentiment_label(example):
    example['sentiment'] = map_star_to_sentiment(example['n_star'])
    return example

dataset = dataset.map(add_sentiment_label)

# Kiểm tra kết quả
print(dataset['train'][0])

In [ ]:
from collections import Counter

sentiments = [item['sentiment'] for item in dataset['train']]
counter = Counter(sentiments)
print("Phân bố nhãn cảm xúc (tập train):")
for sentiment, count in counter.items():
    print(f"  {sentiment}: {count} mẫu")

In [ ]:
import random
from datasets import Dataset

# Đặt seed để kết quả reproducible
random.seed(42)

# Lấy index của từng nhãn
pos_indices = [i for i, item in enumerate(dataset['train']) if item['sentiment'] == 'POSITIVE']
neg_indices = [i for i, item in enumerate(dataset['train']) if item['sentiment'] == 'NEGATIVE']
neu_indices = [i for i, item in enumerate(dataset['train']) if item['sentiment'] == 'NEUTRAL']

print(f"POSITIVE: {len(pos_indices)} mẫu")
print(f"NEGATIVE: {len(neg_indices)} mẫu")
print(f"NEUTRAL:  {len(neu_indices)} mẫu")

# Số mẫu chung = số mẫu ít nhất
n_samples = min(len(pos_indices), len(neg_indices), len(neu_indices))
print(f"\nSố mẫu chung sau undersampling: {n_samples}")

# Chọn ngẫu nhiên
pos_sampled = random.sample(pos_indices, n_samples)
neg_sampled = random.sample(neg_indices, n_samples)
neu_sampled = neu_indices  # NEUTRAL giữ nguyên vì đã ít nhất

# Gộp và xáo trộn
all_indices = pos_sampled + neg_sampled + neu_sampled
random.shuffle(all_indices)

# Tạo tập train mới
train_balanced = dataset['train'].select(all_indices)

print(f"\nTập train sau cân bằng: {len(train_balanced)} mẫu")

In [ ]:
from collections import Counter

sentiments = [item['sentiment'] for item in train_balanced]
counter = Counter(sentiments)
print("Phân bố nhãn sau undersampling:")
for sentiment, count in counter.items():
    print(f"  {sentiment}: {count} mẫu")

In [ ]:
# Lưu tập train đã cân bằng vào dataset
dataset['train'] = train_balanced

print(dataset)

In [ ]:
print(dataset)
print()
print("Các tập có sẵn:", list(dataset.keys()))

In [ ]:
from datasets import DatasetDict

# Bước 1: Chia tập train_balanced thành 70% train + 30% tạm
train_testvalid = train_balanced.train_test_split(test_size=0.3, seed=42)

# Bước 2: Chia tập tạm (30%) thành 15% validation + 15% test
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

# Bước 3: Gộp thành DatasetDict
dataset = DatasetDict({
    'train': train_testvalid['train'],
    'validation': test_valid['train'],
    'test': test_valid['test']
})

print(dataset)

In [ ]:
from collections import Counter

for split in ['train', 'validation', 'test']:
    sentiments = [item['sentiment'] for item in dataset[split]]
    counter = Counter(sentiments)
    print(f"\nPhân bố nhãn tập {split.upper()}:")
    for sentiment in ['POSITIVE', 'NEGATIVE', 'NEUTRAL']:
        print(f"  {sentiment}: {counter.get(sentiment, 0)} mẫu")

In [ ]:
# Lưu dataset vào thư mục tạm để dùng cho bước sau
dataset.save_to_disk('/content/uit-visfd-split')
print("Đã lưu dataset vào /content/uit-visfd-split")

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "vinai/phobert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Đã load tokenizer:", MODEL_NAME)
print("Vocab size:", tokenizer.vocab_size)

In [ ]:
sample_text = "Sản phẩm này rất tốt, pin trâu, màn hình đẹp"
tokens = tokenizer(sample_text, return_tensors="pt")
print(tokens)
print("\nSố token:", len(tokens['input_ids'][0]))

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['comment'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
print(tokenized_dataset)

In [ ]:
# Mapping nhãn chữ → số
label_to_id = {
    'NEGATIVE': 0,
    'NEUTRAL': 1,
    'POSITIVE': 2
}

id_to_label = {v: k for k, v in label_to_id.items()}

print("Mapping nhãn:")
for label, id in label_to_id.items():
    print(f"  {label} → {id}")

In [ ]:
def encode_labels(example):
    example['labels'] = label_to_id[example['sentiment']]
    return example

tokenized_dataset = tokenized_dataset.map(encode_labels)

# Kiểm tra
print("Mẫu đầu tiên sau khi encode:")
print("Comment:", tokenized_dataset['train'][0]['comment'][:80], "...")
print("Sentiment gốc:", tokenized_dataset['train'][0]['sentiment'])
print("Labels số:", tokenized_dataset['train'][0]['labels'])

In [ ]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "vinai/phobert-base"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id_to_label,
    label2id=label_to_id
)

# Đưa model lên GPU
model = model.to("cuda")

print("Đã load model:", MODEL_NAME)
print("Số nhãn:", model.config.num_labels)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
from transformers import TrainingArguments
print("Đã import TrainingArguments")

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/phobert-temp",         # ← Colab local, không phải Drive
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_steps=200,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="no",                         # ← Không lưu checkpoint
    load_best_model_at_end=False,
    report_to="none",
    seed=42
)

print("Output dir:", training_args.output_dir)


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)
print("Đã khởi tạo trainer")

In [ ]:
# Xóa cột 'label' gốc (aspect-based, không dùng nữa)
# Đổi tên cột 'labels' thành 'label' để Trainer nhận diện
def fix_label_column(example):
    example['label'] = example['labels']  # copy giá trị số sang label
    return example

# Áp dụng cho cả 3 tập
tokenized_dataset = tokenized_dataset.map(fix_label_column)

# Xóa cột 'labels' cũ (không cần nữa)
tokenized_dataset = tokenized_dataset.remove_columns(['labels'])

# Kiểm tra
print("Features sau khi sửa:")
print(tokenized_dataset['train'].features)
print()
print("Mẫu đầu tiên:")
print("Comment:", tokenized_dataset['train'][0]['comment'][:60], "...")
print("Label:", tokenized_dataset['train'][0]['label'])
print("Sentiment gốc:", tokenized_dataset['train'][0]['sentiment'])

In [ ]:
# Xóa các cột không cần thiết cho training
columns_to_keep = ['input_ids', 'attention_mask', 'label']
columns_to_remove = [col for col in tokenized_dataset['train'].column_names
                     if col not in columns_to_keep]

print("Cột sẽ xóa:", columns_to_remove)

tokenized_dataset = tokenized_dataset.remove_columns(columns_to_remove)

# Kiểm tra
print("\nFeatures sau khi xóa:")
print(tokenized_dataset['train'].features)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

print("Đã khởi tạo lại Trainer")

In [ ]:
trainer.train()


In [ ]:
test_results = trainer.evaluate(tokenized_dataset['test'])
print("Kết quả trên tập test:")
for key, value in test_results.items():
    if key.startswith('eval_'):
        if isinstance(value, float):
            print(f"  {key[5:]}: {value:.4f}")
        else:
            print(f"  {key[5:]}: {value}")

In [ ]:
trainer.save_model("./phobert-finetuned-final")
tokenizer.save_pretrained("./phobert-finetuned-final")
print("Đã lưu model vào ./phobert-finetuned-final")

In [ ]:
from transformers import pipeline

# Load model đã fine-tune
classifier = pipeline(
    "text-classification",
    model="./phobert-finetuned-final",
    tokenizer="./phobert-finetuned-final",
    device=0
)

# Test 3 câu
test_reviews = [
    "Sản phẩm rất tốt, pin trâu, màn hình đẹp, giao hàng nhanh",
    "Hàng kém chất lượng, giao chậm, thất vọng",
    "Bình thường, không có gì đặc biệt"
]

for review in test_reviews:
    result = classifier(review)[0]
    print(f"Review: {review}")
    print(f"  → {result['label']} (score: {result['score']:.4f})")
    print()

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Đổi base model
MODEL_NAME = "mr4/phobert-base-vi-sentiment-analysis"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id_to_label,
    label2id=label_to_id,
    ignore_mismatched_sizes=True
)

model = model.to("cuda")

print("Đã load model:", MODEL_NAME)
print("Số nhãn:", model.config.num_labels)

In [ ]:
from datasets import load_dataset, DatasetDict

# Load lại dataset gốc
raw_dataset = load_dataset("visolex/ViSFD")

# Mapping nhãn cảm xúc
def add_sentiment_label(example):
    n = example['n_star']
    if n <= 2:
        example['sentiment'] = 'NEGATIVE'
    elif n == 3:
        example['sentiment'] = 'NEUTRAL'
    else:
        example['sentiment'] = 'POSITIVE'
    return example

raw_dataset = raw_dataset.map(add_sentiment_label)

# Chia tập: 70% train, 15% val, 15% test (KHÔNG undersample)
train_testvalid = raw_dataset['train'].train_test_split(test_size=0.3, seed=42)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

dataset_full = DatasetDict({
    'train': train_testvalid['train'],
    'validation': test_valid['train'],
    'test': test_valid['test']
})

print("Số mẫu từng tập:")
print(f"  Train: {len(dataset_full['train'])}")
print(f"  Validation: {len(dataset_full['validation'])}")
print(f"  Test: {len(dataset_full['test'])}")

In [ ]:
!ls /content/drive/MyDrive/InsightAI/

In [ ]:
print(training_args.output_dir)

In [ ]:
trainer.train()


In [ ]:
print("Số epoch:", training_args.num_train_epochs)
print("Output dir:", training_args.output_dir)

In [ ]:
print("Model name:", model.config.name_or_path)

In [ ]:
from transformers import pipeline
import random

# Load model đã fine-tune lần 2
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0
)

# Lấy ngẫu nhiên 5 review từ tập test
random.seed(42)
sample_indices = random.sample(range(len(dataset_full['test'])), 5)

print("=" * 80)
print("TEST 5 REVIEW THỰC TỪ TẬP TEST")
print("=" * 80)

label_map = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}

for i, idx in enumerate(sample_indices):
    item = dataset_full['test'][idx]
    review = item['comment']
    true_label = item['sentiment']

    # Dự đoán
    result = classifier(review[:512])[0]  # Giới hạn 512 ký tự
    pred_label = result['label']
    score = result['score']

    # So sánh
    match = "✅ ĐÚNG" if pred_label == true_label else "❌ SAI"

    print(f"\n--- Review {i+1} ---")
    print(f"Nội dung: {review[:200]}{'...' if len(review) > 200 else ''}")
    print(f"Nhãn thật:  {true_label}")
    print(f"Dự đoán:    {pred_label} (score: {score:.4f})")
    print(f"Kết quả:    {match}")

print("\n" + "=" * 80)

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="mr4/phobert-base-vi-sentiment-analysis"
)

test_reviews = [
    "Quá chán, quá thất vọng. Pin nhanh hết",
    "Máy dùng mượt, chơi game không bị lag",
    "Máy dùng rất tốt khá ok",
    "Điện thoại thiết kế đẹp, chụp hình rõ nét",
    "Máy bắt sóng sim yếu"
]

for review in test_reviews:
    result = classifier(review)[0]
    print(f"{review} → {result['label']} ({result['score']:.4f})")